# Sphero BOLT+ — Physics Sim2Real

Single notebook for the full pipeline:
1. Load real-robot recordings
2. Visualize raw & derived speed/acceleration signals
3. Grid-search `(max_speed_ms, max_accel_ms2)` to minimize sim2real distance RMSE
4. Compare simulated vs measured traces with the best parameters
5. Analyze command-to-motion delay

| Module            | Responsibility                                                                             |
|-------------------|--------------------------------------------------------------------------------------------|
| `speed_data.py`   | **One CSV loader** — constants, `prepare_run_df`, `load_all_runs`, `estimate_motion_delay` |
| `calib_data.py`   | `RunMetrics` dataclass + `df_to_run_metrics` (wraps `speed_data`)                          |
| `calib_sim.py`    | Pure-Python sim via `SpheroController.get_velocity()`                                      |
| `calib_search.py` | Grid search, range builder, JSON export                                                    |
| `calib_plot.py`   | 8-panel calibration figure                                                                 |
| `speed_plot.py`   | Speed-analysis plot functions                                                              |

In [ ]:
# ── Path setup ────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

_here = Path('__file__').resolve().parent
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

# ── Imports ───────────────────────────────────────────────────────────────────
from speed_data  import load_all_runs
from calib_data  import df_to_run_metrics
from calib_search import build_ranges, grid_search, save_best_params, plot_speed_distance_factor, load_grid_results
from calib_plot  import plot_results, print_metrics_table, plot_panel,  print_metrics_table_with_sim
from speed_plot  import (
    plot_single_speed,
    plot_all_speeds,
    plot_single_speed_with_sim,
    plot_all_distance_with_sim_subplots,
    plot_delay_vs_speed,
)

print('Imports OK')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                    ── CONFIG CELL ──                         ║
# ║  Edit here only.  All other cells run without changes.       ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Data source ───────────────────────────────────────────────────────────────
DATA_DIR       = 'csv_files/data_fit'  # directory with exp_speed_*.csv files
SPEED_COL      = 'cmd'             # speed-command column (combined-CSV mode only)

# ── Signal preprocessing ──────────────────────────────────────────────────────
REDUCTION_MODE = 'center'      # 'none' | 'first_last' | 'center' | 'first' | 'last'
WINDOW         = 1                 # rolling-mean window for derived speed
SMOOTH_WINDOW  = 1                 # uniform_filter1d window for RunMetrics (calibration)

# ── Error metric ──────────────────────────────────────────────────────────────
USE_PERCENT_ERROR = True   # False = absolute RMSE (m) | True = RMSPE (%)
RMSE_UNIT = "%" if USE_PERCENT_ERROR else "m"

# ── Single-run selector ───────────────────────────────────────────────────────
SELECTED_SPEED = 50                # speed command to inspect in detail

# ── Grid search bounds ────────────────────────────────────────────────────────
SPEED_MIN,  SPEED_MAX,  SPEED_STEPS = 0.0, 5.0, 5 +1   # max_speed_ms  (m/s)
ACCEL_MIN,  ACCEL_MAX,  ACCEL_STEPS = 0.0, 5.0, 5 +1  # max_accel_ms2 (m/s²)

SPEED_MIN_SAPIEN,  SPEED_MAX_SAPIEN,  SPEED_STEPS_SAPIEN = 0.0, 5.0, 5 +1   # max_speed_ms  (m/s)
ACCEL_MIN_SAPIEN,  ACCEL_MAX_SAPIEN,  ACCEL_STEPS_SAPIEN = 0.0, 5.0, 5 +1  # max_accel_ms2 (m/s²)

# ── Grid search cache ─────────────────────────────────────────────────────────
# To force a re-run of the grid search (e.g. after changing bounds), just delete the .npz file or pass cache_path=None.
GRID_CACHE        = "grid_cache.npz"         # pure-Python sim cache
GRID_CACHE_SAPIEN = "grid_cache_sapien.npz"  # SAPIEN sim cache

# ── Outputs ───────────────────────────────────────────────────────────────────
OUT_FIGURE = 'calibration_results.png'
OUT_PARAMS = 'best_params.json'

print('Config OK')

In [ ]:
# ── Load data (once — shared by all sections) ─────────────────────────────────
#
# runs_df   : dict[int, DataFrame]  — full time-series for plotting
# runs_metrics : list[RunMetrics]   — compact metrics for grid search

runs_df = load_all_runs(DATA_DIR, reduction_mode=REDUCTION_MODE, window=WINDOW)
runs_metrics = [
    df_to_run_metrics(df, speed_cmd, smooth_window=SMOOTH_WINDOW)
    for speed_cmd, df in sorted(runs_df.items())
]

print(f'Speed commands : {sorted(runs_df)}')
print(f'RunMetrics     : {[r.speed_cmd for r in runs_metrics]}')

## Real Robot Metrics

In [ ]:
print_metrics_table(runs_metrics)

In [ ]:
factor = plot_speed_distance_factor(runs_metrics)

## Speed Analysis — Single Run

In [ ]:
plot_single_speed(SELECTED_SPEED, runs_df, mode='raw',     show_components=True)
plot_single_speed(SELECTED_SPEED, runs_df, mode='derived', show_components=True)

## Speed Analysis — All Speeds

In [ ]:
plot_all_speeds(runs_df, mode='raw')
plot_all_speeds(runs_df, mode='derived')

## Calibration — Grid Search

In [ ]:
max_speed_range, max_accel_range = build_ranges(
    SPEED_MIN, SPEED_MAX, SPEED_STEPS,
    ACCEL_MIN, ACCEL_MAX, ACCEL_STEPS,
)
print(f"Max Speed Range: {max_speed_range[0]} - {max_speed_range[-1]}  | Steps: {max_speed_range[1] - max_speed_range[0]}")
print(f"Max Accel Range: {max_accel_range[0]} - {max_accel_range[-1]}  | Steps: {max_accel_range[1] - max_accel_range[0]}")
# max_accel_range, max_speed_range

In [ ]:
best_speed, best_accel, rmse_grid = grid_search(runs_metrics, max_speed_range, max_accel_range, use_percent_error=USE_PERCENT_ERROR, cache_path=GRID_CACHE)

save_best_params(
    OUT_PARAMS,
    best_speed, best_accel,
    best_rmse=float(rmse_grid.min()),
    grid_config={
        'speed_range': [SPEED_MIN, SPEED_MAX, SPEED_STEPS],
        'accel_range': [ACCEL_MIN, ACCEL_MAX, ACCEL_STEPS],
    },
    use_percent_error=USE_PERCENT_ERROR,
)

print(f'\nPlug into SpheroController(')
print(f'    max_speed_ms  = {best_speed:.4f},')
print(f'    max_accel_ms2 = {best_accel:.4f},')
print(f')')
print_metrics_table_with_sim(runs_metrics, best_speed, best_accel)

## Calibration Results

In [ ]:
plot_results(
    runs_metrics,
    best_speed,
    best_accel,
    rmse_grid,
    max_speed_range,
    max_accel_range,
    out_path=OUT_FIGURE,
    rmse_unit=RMSE_UNIT,
)

## Sim vs Real

In [ ]:
SELECTED_SPEED = 75
plot_single_speed_with_sim(SELECTED_SPEED, runs_df, best_speed, best_accel)

In [ ]:
plot_all_distance_with_sim_subplots(runs_df, best_speed, best_accel, cols=3, share_y=False)
#plot_all_distance_with_sim_subplots(runs_df, 2, 4, cols=3)

## Motion Delay

In [ ]:
plot_delay_vs_speed(runs_df)

# Sapien

In [ ]:
from calib_sapien import grid_search_sapien
from calib_search import build_ranges, save_best_params

In [ ]:
speed_range, accel_range = build_ranges(
    SPEED_MIN_SAPIEN, SPEED_MAX_SAPIEN, SPEED_STEPS_SAPIEN,
    ACCEL_MIN_SAPIEN, ACCEL_MAX_SAPIEN, ACCEL_STEPS_SAPIEN,
)
print(f"speed_range: {speed_range[0]:.2f} → {speed_range[-1]:.2f}, step={speed_range[1]-speed_range[0]:.4f}, N={len(speed_range)}")
print(f"accel_range: {accel_range[0]:.2f} → {accel_range[-1]:.2f}, step={accel_range[1]-accel_range[0]:.4f}, N={len(accel_range)}")

In [ ]:
from calib_sapien import make_simulate_fn

sim_fn = make_simulate_fn()
best_speed, best_accel, rmse_grid = grid_search_sapien(
    runs_metrics,                # same runs list from your existing section
    speed_range,
    accel_range,
    use_percent_error=USE_PERCENT_ERROR,
    cache_path=GRID_CACHE_SAPIEN
)
# Uses SAPIEN engine for sim column
print_metrics_table_with_sim(runs_metrics, best_speed, best_accel, simulate_fn=sim_fn)

In [ ]:
import numpy as np
save_best_params(
    path="best_params_sapien.json",
    best_speed=best_speed,
    best_accel=best_accel,
    best_rmse=float(np.min(rmse_grid)),
    grid_config={
        'speed_range': [SPEED_MIN_SAPIEN, SPEED_MAX_SAPIEN, SPEED_STEPS_SAPIEN],
        'accel_range': [ACCEL_MIN_SAPIEN, ACCEL_MAX_SAPIEN, ACCEL_STEPS_SAPIEN],
    },
)

In [ ]:
# ── Sapien: same plots as above but using SAPIEN physics ─────────────────────


plot_results(
    runs_metrics,
    best_speed, best_accel,
    rmse_grid,
    speed_range, accel_range,
    out_path="calibration_results_sapien.png",
    rmse_unit="m",
    simulate_fn=sim_fn,
)

In [ ]:
SELECTED_SPEED = 75
plot_single_speed_with_sim(SELECTED_SPEED, runs_df, best_speed, best_accel, simulate_fn=sim_fn)

In [ ]:
plot_all_distance_with_sim_subplots(runs_df, best_speed, best_accel, cols=3, share_y=False, simulate_fn=sim_fn)

In [ ]:
# Close engine when done
#sim_fn.engine.close()

In [ ]:
# ── Export individual panels for thesis ───────────────────────────────────────
plot_panel("rmse_heatmap", runs_metrics, best_speed, best_accel,
           rmse_grid=rmse_grid, max_speed_range=max_speed_range,
           max_accel_range=max_accel_range,
           rmse_unit=RMSE_UNIT, out_path="thesis_rmse_heatmap.png", figsize=(7, 5))

plot_panel("distance", runs_metrics, best_speed, best_accel,
           out_path="thesis_distance.png")

plot_panel("accel_decel", runs_metrics, best_speed, best_accel,
           out_path="thesis_accel_decel.png", figsize=(10, 5))

plot_panel("gap", runs_metrics, best_speed, best_accel,
           out_path="thesis_gapl.png", figsize=(10, 5))